# FreshIQ — Avocado Ripening Model V0.1

**Goal:** Train a leakage-safe 5-class computer-vision model to predict avocado ripening stage from an RGB image.

### Dataset
- 14,710 usable JPG images
- 478 experimental specimens
- 5 ripening stages
- Specimen-level train/validation/test split
- 12 metadata rows without a matching image are excluded

### Model
MobileNetV3-Small pretrained on ImageNet, with the final classifier replaced by a 5-class head.

> **Important:** The dataset must be available locally. Update `DATA_ROOT` and `MANIFEST_DIR` in the configuration cell if your paths differ.

In [1]:
# 1. Imports and reproducibility

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Device:", DEVICE)

ModuleNotFoundError: No module named 'torch'

In [ ]:
# 2. Configuration
#
# CHANGE THESE TWO PATHS to match your computer.

DATA_ROOT = Path(r"./dataset")       # Folder containing the extracted avocado images
MANIFEST_DIR = Path(r"./manifests")  # Folder containing train.csv / validation.csv / test.csv

TRAIN_CSV = MANIFEST_DIR / "train.csv"
VAL_CSV = MANIFEST_DIR / "validation.csv"
TEST_CSV = MANIFEST_DIR / "test.csv"

BATCH_SIZE = 32
NUM_WORKERS = 2
NUM_CLASSES = 5
NUM_EPOCHS = 12
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

MODEL_DIR = Path("./models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = MODEL_DIR / "freshiq_mobilenetv3_avocado_best.pth"

print("Train CSV:", TRAIN_CSV.resolve())
print("Validation CSV:", VAL_CSV.resolve())
print("Test CSV:", TEST_CSV.resolve())

## 3. Load the manifests

The manifests were generated using **specimen-level splitting**.

That means all observations/images belonging to one experimental avocado stay in exactly one split, preventing the same avocado from appearing in both training and testing.

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

# Only use rows whose image actually exists.
for name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(f"{name:12s}: {len(df):,} rows")

print("\nColumns:")
print(train_df.columns.tolist())

print("\nUnique specimens:")
print("train:", train_df.specimen_id.nunique())
print("val:  ", val_df.specimen_id.nunique())
print("test: ", test_df.specimen_id.nunique())

# Safety check: no specimen leakage.
assert set(train_df.specimen_id).isdisjoint(set(val_df.specimen_id))
assert set(train_df.specimen_id).isdisjoint(set(test_df.specimen_id))
assert set(val_df.specimen_id).isdisjoint(set(test_df.specimen_id))

print("\nLeakage check: PASSED")

In [ ]:
# 4. Dataset statistics

stage_names = {
    1: "Stage 1",
    2: "Stage 2",
    3: "Stage 3",
    4: "Stage 4",
    5: "Stage 5",
}

print("Training class distribution:")
print(train_df["ripening_stage"].value_counts().sort_index())

print("\nValidation class distribution:")
print(val_df["ripening_stage"].value_counts().sort_index())

print("\nTest class distribution:")
print(test_df["ripening_stage"].value_counts().sort_index())

# Visualize training distribution.
counts = train_df["ripening_stage"].value_counts().sort_index()

plt.figure(figsize=(8, 5))
plt.bar([stage_names[i] for i in counts.index], counts.values)
plt.xlabel("Ripening stage")
plt.ylabel("Number of images")
plt.title("FreshIQ training-set class distribution")
plt.tight_layout()
plt.show()

## 5. Verify image paths

The manifest stores paths relative to `DATA_ROOT`.

This cell checks that the actual images can be opened before training.

In [ ]:
def resolve_image_path(relative_path):
    return DATA_ROOT / str(relative_path)

def check_images(df, limit=None):
    subset = df if limit is None else df.head(limit)
    missing = []
    unreadable = []

    for rel in subset["image_path"].dropna():
        path = resolve_image_path(rel)

        if not path.exists():
            missing.append(str(path))
            continue

        try:
            with Image.open(path) as img:
                img.verify()
        except Exception:
            unreadable.append(str(path))

    return missing, unreadable

for name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    missing, unreadable = check_images(df)
    print(f"{name}: missing={len(missing)}, unreadable={len(unreadable)}")

## 6. Visualize examples

This is a sanity check only. The model receives the raw RGB image; metadata such as day and storage group are not used by this first model.

In [ ]:
def show_examples(df, n_per_stage=3):
    fig, axes = plt.subplots(5, n_per_stage, figsize=(4*n_per_stage, 18))

    for stage in range(1, 6):
        pool = df[df.ripening_stage == stage]
        samples = pool.sample(min(n_per_stage, len(pool)), random_state=SEED)

        for j, (_, row) in enumerate(samples.iterrows()):
            ax = axes[stage-1, j] if n_per_stage > 1 else axes[stage-1]
            path = resolve_image_path(row.image_path)

            with Image.open(path) as img:
                ax.imshow(img.convert("RGB"))

            ax.set_title(
                f"Stage {stage}\n{row.specimen_id}, Day {int(row.day)}"
            )
            ax.axis("off")

    plt.tight_layout()
    plt.show()

show_examples(train_df)

## 7. Image transforms

We use ImageNet normalization because the backbone is pretrained on ImageNet.

Training gets mild augmentation. Validation and testing use deterministic preprocessing.

In [ ]:
weights = MobileNet_V3_Small_Weights.DEFAULT

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
        hue=0.02,
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

## 8. PyTorch Dataset

Images are loaded **on demand**, so the full 14,710-image dataset does not need to fit into RAM.

In [ ]:
class AvocadoDataset(Dataset):
    def __init__(self, dataframe, root, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.root = Path(root)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = self.root / str(row["image_path"])

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        # Convert stages 1-5 to class indices 0-4.
        label = int(row["ripening_stage"]) - 1

        return image, label


train_ds = AvocadoDataset(train_df, DATA_ROOT, train_transform)
val_ds = AvocadoDataset(val_df, DATA_ROOT, eval_transform)
test_ds = AvocadoDataset(test_df, DATA_ROOT, eval_transform)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

## 9. Load MobileNetV3-Small

The model starts from ImageNet pretrained weights and replaces the final classifier with a 5-class output.

If the pretrained weights cannot be downloaded, **stop here** and fix the network/cache rather than silently training an uninitialized model.

In [ ]:
model = mobilenet_v3_small(weights=weights)

# Freeze the convolutional backbone for the first baseline.
for param in model.features.parameters():
    param.requires_grad = False

in_features = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(in_features, NUM_CLASSES)

model = model.to(DEVICE)

print(model.classifier)

## 10. Loss, optimizer and training helpers

We use weighted cross-entropy based on the training distribution so minority stages are not ignored.

In [ ]:
stage_counts = train_df["ripening_stage"].value_counts().sort_index()
class_counts = np.array([stage_counts.get(i, 0) for i in range(1, 6)], dtype=np.float32)

class_weights = class_counts.sum() / (NUM_CLASSES * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

print("Class weights:", class_weights.detach().cpu().numpy())

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    y_true = []
    y_pred = []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = criterion(logits, labels)

        if is_train:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * images.size(0)

        predictions = logits.argmax(dim=1)
        y_true.extend(labels.detach().cpu().numpy())
        y_pred.extend(predictions.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")

    return avg_loss, accuracy, f1

## 11. Train Model V0.1

The best checkpoint is selected using **validation macro-F1**, not test performance.

In [ ]:
history = []
best_val_f1 = -1.0

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc, train_f1 = run_epoch(
        model, train_loader, criterion, optimizer
    )

    with torch.no_grad():
        val_loss, val_acc, val_f1 = run_epoch(
            model, val_loader, criterion
        )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "train_f1": train_f1,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "val_f1": val_f1,
    })

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({
            "model_state_dict": model.state_dict(),
            "class_names": [stage_names[i] for i in range(1, 6)],
            "model_name": "mobilenet_v3_small",
            "seed": SEED,
        }, BEST_MODEL_PATH)

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"train loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
        f"val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
    )

print("\nBest validation macro-F1:", best_val_f1)
print("Saved:", BEST_MODEL_PATH)

In [ ]:
# 12. Training curves

history_df = pd.DataFrame(history)

plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_loss"], label="Train loss")
plt.plot(history_df["epoch"], history_df["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("FreshIQ training and validation loss")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_f1"], label="Train macro-F1")
plt.plot(history_df["epoch"], history_df["val_f1"], label="Validation macro-F1")
plt.xlabel("Epoch")
plt.ylabel("Macro-F1")
plt.title("FreshIQ training and validation macro-F1")
plt.legend()
plt.tight_layout()
plt.show()

## 13. Final test evaluation

The test set is used **once for final evaluation** after selecting the best checkpoint using validation macro-F1.

In [ ]:
checkpoint = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

all_true = []
all_pred = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        logits = model(images)
        preds = logits.argmax(dim=1)

        all_true.extend(labels.numpy())
        all_pred.extend(preds.cpu().numpy())

test_accuracy = accuracy_score(all_true, all_pred)
test_macro_f1 = f1_score(all_true, all_pred, average="macro")

print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test macro-F1: {test_macro_f1:.4f}")

print("\nClassification report:")
print(
    classification_report(
        all_true,
        all_pred,
        labels=[0,1,2,3,4],
        target_names=[stage_names[i] for i in range(1,6)],
        digits=4,
        zero_division=0,
    )
)

In [ ]:
# 14. Confusion matrix

cm = confusion_matrix(all_true, all_pred, labels=[0,1,2,3,4])

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[stage_names[i] for i in range(1,6)]
)

fig, ax = plt.subplots(figsize=(8, 7))
disp.plot(ax=ax, values_format="d")
ax.set_title("FreshIQ avocado ripening-stage confusion matrix")
plt.tight_layout()
plt.show()

## 15. Save experiment results

This records the main metrics and configuration so later model versions can be compared fairly.

In [ ]:
results = {
    "model": "MobileNetV3-Small",
    "pretrained": True,
    "num_classes": NUM_CLASSES,
    "train_specimens": int(train_df.specimen_id.nunique()),
    "validation_specimens": int(val_df.specimen_id.nunique()),
    "test_specimens": int(test_df.specimen_id.nunique()),
    "test_accuracy": float(test_accuracy),
    "test_macro_f1": float(test_macro_f1),
    "best_validation_macro_f1": float(best_val_f1),
    "batch_size": BATCH_SIZE,
    "epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "seed": SEED,
}

results_path = MODEL_DIR / "baseline_results.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))

# Model V0.1 complete

If this baseline performs reasonably well, the next experiment should **not** immediately jump to a bigger model.

Next we will compare:

1. Frozen MobileNetV3 baseline
2. Fine-tuned MobileNetV3
3. Optional metadata features (day/storage condition)
4. Remaining-shelf-life model using the longitudinal trajectories

The final FreshIQ architecture should only be chosen after these experiments.